# LangChain: Agents (Groq Llama 3.1)

## Outline
* **Built-in Tools** - DuckDuckGo search, Wikipedia
* **Python Agent** - Execute Python code dynamically
* **Custom Tools** - Define your own tools with @tool decorator
* **Debug Mode** - Inspect agent reasoning process

**Model Used:** Groq Llama-3.1-8b-instant

**Key Concept:** LLMs as **reasoning engines**, not just knowledge stores. Agents decide which tools to use and when to use them.

**What is an Agent?**
An agent uses an LLM to decide which actions to take and in what order. Unlike chains (predetermined sequence), agents dynamically choose their path based on inputs.


In [9]:
# Cell 2: Install Required Packages

!pip install langchain langchain-groq python-dotenv wikipedia ddgs -q


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Cell 3: Import Libraries and Load Environment

import os
import warnings
from dotenv import load_dotenv, find_dotenv

# Load environment variables
load_dotenv(find_dotenv())

# Suppress warnings
warnings.filterwarnings("ignore")

# Verify API key
groq_api_key = os.environ.get('GROQ_API_KEY')
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in .env file!")
    
print("✅ Environment loaded successfully")


✅ Environment loaded successfully


In [6]:
# Cell 4: Import Agent Components

from langchain_groq import ChatGroq
from langchain_experimental.tools import PythonREPLTool
from langchain_community.tools import WikipediaQueryRun, DuckDuckGoSearchRun
from langchain_community.utilities import WikipediaAPIWrapper
from langgraph.prebuilt import create_react_agent

print("✅ Agent components imported")

✅ Agent components imported


In [7]:
# Cell 5: Initialize LLM

# Set model
llm_model = "llama-3.1-8b-instant"

# Initialize Groq LLM
llm = ChatGroq(temperature=0, model=llm_model, groq_api_key=groq_api_key)

print(f"✅ Model initialized: {llm_model}")


✅ Model initialized: llama-3.1-8b-instant


In [16]:
# Cell 6: Load Built-in Tools

from langchain_core.tools import tool

# Create a simple calculator tool
@tool
def calculator(expression: str) -> str:
    """Useful for performing mathematical calculations. 
    Input should be a valid mathematical expression like '25 * 300 / 100' or '0.25 * 300'."""
    try:
        # Safely evaluate mathematical expressions
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error calculating: {str(e)}"

# Initialize Wikipedia tool
wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

# Initialize DuckDuckGo search tool
search = DuckDuckGoSearchRun()

# Create tools list
tools = [calculator, wikipedia, search]

print(f"✅ Loaded {len(tools)} tools:")
for tool_item in tools:
    print(f"  - {tool_item.name}")

✅ Loaded 3 tools:
  - calculator
  - wikipedia
  - duckduckgo_search


In [17]:
# Cell 7: Create Agent

# Create the ReAct agent using LangGraph (modern approach)
agent_executor = create_react_agent(llm, tools)

print("✅ Agent created successfully")
print("\nAgent Type: ReAct (Reasoning + Acting)")
print("  - Uses chain-of-thought reasoning")
print("  - Dynamically selects and uses tools")
print("  - Iterates until answer is found")

✅ Agent created successfully

Agent Type: ReAct (Reasoning + Acting)
  - Uses chain-of-thought reasoning
  - Dynamically selects and uses tools
  - Iterates until answer is found


In [18]:
# Cell 8: Test Math Tool

# Simple math question
response = agent_executor.invoke({"messages": [("user", "What is 25% of 300?")]})

print("\nFinal Answer:")
print(response['messages'][-1].content)


Final Answer:
The result of the calculation is 75.0.


In [19]:
# Cell 9: Test Wikipedia Tool

question = "Tom M. Mitchell is an American computer scientist \
and the Founders University Professor at Carnegie Mellon University (CMU) \
what book did he write?"

result = agent_executor.invoke({"messages": [("user", question)]})

print("\nFinal Answer:")
print(result['messages'][-1].content)


Final Answer:
Tom M. Mitchell wrote the textbook "Machine Learning".


In [20]:
# Cell 10: Test Current Events Question

# Ask about 2022 World Cup
question = "Who won the 2022 FIFA World Cup?"

try:
    result = agent_executor.invoke({"messages": [("user", question)]})
    print("\nFinal Answer:")
    print(result['messages'][-1].content)
except Exception as e:
    print(f"Agent encountered an issue: {e}")
    print("\n⚠️ Agents are experimental and may not always succeed")

Agent encountered an issue: Error code: 400 - {'error': {'message': "tool call validation failed: attempted to call tool 'brave_search' which was not in request.tools", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=brave_search>{"query": "2022 FIFA World Cup winner"}</function>'}}

⚠️ Agents are experimental and may not always succeed


## Python Agent

The **Python Agent** can write and execute Python code to solve problems. This is powerful for:
- Data manipulation
- Mathematical calculations
- List/array operations
- Any task that's easier to solve with code than natural language


In [21]:
# Cell 11: Create Python Agent

# Create Python REPL tool
python_repl = PythonREPLTool()

# Create Python ReAct agent using LangGraph
python_agent_executor = create_react_agent(llm, [python_repl])

print("✅ Python Agent created")

✅ Python Agent created


In [22]:
# Cell 12: Test Python Agent - Sort Customer List

# Sample customer data
customer_list = [
    ["Harrison", "Chase"], 
    ["Lang", "Chain"],
    ["Dolly", "Too"],
    ["Elle", "Elem"], 
    ["Geoff", "Fusion"], 
    ["Trance", "Former"],
    ["Jen", "Ayai"]
]

# Ask agent to sort customers
result = python_agent_executor.invoke({
    "messages": [("user", f"Sort these customers by last name and then first name and print the output: {customer_list}")]
})

print("\nResult:")
print(result['messages'][-1].content)

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=Python_REPL>{query: ["[\'Harrison\', \'Chase\']", "[\'Lang\', \'Chain\']", "[\'Dolly\', \'Too\']", "[\'Elle\', \'Elem\']", "[\'Geoff\', \'Fusion\']", "[\'Trance\', \'Former\']", "[\'Jen\', \'Ayai\']"].sort(key=lambda x: (x[0], x[1]))} </function>'}}

## Debug Mode

Debug mode shows the agent's thought process:
- Which tool it chooses
- Why it chose that tool
- The input to the tool
- The tool's output
- The agent's reasoning


In [23]:
# Cell 13: Enable Debug Mode

import langchain
langchain.debug = True

print("✅ Debug mode enabled - you'll see detailed agent reasoning")


✅ Debug mode enabled - you'll see detailed agent reasoning


In [ ]:
# Cell 14: Run Agent with Debug Output

# Run the same sorting task with debug enabled
result = python_agent_executor.invoke({
    "messages": [("user", f"Sort these customers by last name and then first name and print the output: {customer_list}")]
})

In [24]:
# Cell 15: Disable Debug Mode

langchain.debug = False

print("✅ Debug mode disabled")


✅ Debug mode disabled


## Define Your Own Tool

You can create custom tools to connect agents to:
- Your own APIs
- Internal databases
- Custom functions
- Any external service

**Steps:**
1. Use the `@tool` decorator
2. Write a descriptive docstring (agent uses this!)
3. Define the function logic


In [27]:
# Cell 16: Import Tool Decorator

from langchain_core.tools import tool
from datetime import date

print("✅ Tool decorator imported")

✅ Tool decorator imported


In [26]:
# Cell 17: Create Custom Time Tool

@tool
def time(text: str) -> str:
    """Returns todays date, use this for any \
    questions related to knowing todays date. \
    The input should always be an empty string, \
    and this function will always return todays \
    date - any date mathmatics should occur \
    outside this function."""
    return str(date.today())

print("✅ Custom 'time' tool created")
print(f"Tool name: {time.name}")
print(f"Tool description: {time.description}")


✅ Custom 'time' tool created
Tool name: time
Tool description: Returns todays date, use this for any     questions related to knowing todays date.     The input should always be an empty string,     and this function will always return todays     date - any date mathmatics should occur     outside this function.


In [28]:
# Cell 18: Create Agent with Custom Tool

# Create agent with built-in tools + custom time tool
custom_tools = tools + [time]

# Create ReAct agent with custom tools using LangGraph
custom_agent_executor = create_react_agent(llm, custom_tools)

print("✅ Agent created with custom time tool")

✅ Agent created with custom time tool


In [29]:
# Cell 19: Test Custom Tool

# Ask about today's date
result = custom_agent_executor.invoke({"messages": [("user", "What is today's date?")]})

print("\nFinal Answer:")
print(result['messages'][-1].content)


Final Answer:
The current date is 2026-02-11.


In [30]:
# Cell 20: Test Complex Query with Multiple Tools

# Question requiring both Wikipedia and date tools
question = "When was Python programming language created? \
How many years ago was that from today?"

result = custom_agent_executor.invoke({"messages": [("user", question)]})

print("\nFinal Answer:")
print(result['messages'][-1].content)

BadRequestError: Error code: 400 - {'error': {'message': "tool call validation failed: parameters for tool time did not match schema: errors: [missing properties: 'text']", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=time>{}</function>\n'}}

## Important Notes on Agents

### ⚠️ Agents Are Experimental
- May not always reach the correct conclusion
- Can make mistakes in reasoning
- May call tools in unexpected order
- If it fails, try running again

### Why Agents Fail:
1. **Tool selection errors** - Chooses wrong tool
2. **Parsing errors** - Can't parse LLM output correctly
3. **Context limitations** - Loses track in long chains
4. **Ambiguous questions** - Unclear what to do

### Best Practices:
✅ Write clear, specific questions  
✅ Use verbose=True during development  
✅ Set handle_parsing_errors=True  
✅ Provide detailed tool descriptions  
✅ Test with multiple runs  


In [31]:
# Cell 22: Create Another Custom Tool Example

@tool
def multiply(a: str, b: str) -> str:
    """Multiply two numbers together. \
    The input should be two numbers separated by a comma. \
    For example: '5,3' or '10,20'."""
    try:
        nums = a.split(',')
        return str(float(nums[0]) * float(nums[1]))
    except:
        return "Error: Please provide two numbers separated by a comma"

print("✅ Custom 'multiply' tool created")


✅ Custom 'multiply' tool created


In [ ]:
# Cell 23: Test Multiple Custom Tools

# Create agent with all tools including multiply
all_tools = tools + [time, multiply]

# Create ReAct agent with all custom tools using LangGraph
agent_with_custom_tools = create_react_agent(llm, all_tools)

# Test the multiply tool
result = agent_with_custom_tools.invoke({"messages": [("user", "What is 123 multiplied by 456?")]})

print("\nFinal Answer:")
print(result['messages'][-1].content)

## How Agents Work: The ReAct Loop

### ReAct = Reasoning + Acting

**Step 1: Thought**
- Agent analyzes the question
- Decides which tool to use

**Step 2: Action**
- Selects a tool
- Determines the input for that tool

**Step 3: Observation**
- Tool executes and returns result
- Agent observes the output

**Step 4: Repeat or Answer**
- If more info needed → go to Step 1
- If enough info → generate final answer

### Example Flow:
Question: "What's the date and what's 25% of 300?"

Thought: I need today's date first
Action: Use 'time' tool
Observation: 2026-02-11

Thought: Now I need to calculate 25% of 300
Action: Use 'Calculator' tool with "300 * 0.25"
Observation: 75.0

Thought: I have both pieces of information
Final Answer: Today is 2026-02-11 and 25% of 300 is 75.

## Summary: LangChain Agents

### What We Learned:

1. **Agents vs Chains**
   - **Chains**: Predetermined sequence of steps
   - **Agents**: Dynamically decide which tools to use

2. **Built-in Tools**
   - Wikipedia (entity/person information)
   - Calculator (math operations)
   - Python REPL (execute code)
   - DuckDuckGo (web search)

3. **Custom Tools**
   - Use `@tool` decorator
   - Write detailed docstrings
   - Agent uses description to decide when to use it

4. **Agent Types**
   - `CHAT_ZERO_SHOT_REACT_DESCRIPTION`: Best for chat models
   - Others: STRUCTURED_CHAT, CONVERSATIONAL, etc.

### Agent Architecture:
User Question
↓
Agent (LLM as reasoning engine)
↓
Choose Tool
↓
Execute Tool
↓
Observe Result
↓
More tools needed? → Yes → Loop back
↓ No
Final Answer


### Use Cases:

- **Research Assistant** - Search + summarize information
- **Data Analysis** - Query databases + run calculations  
- **API Interactions** - Call external services dynamically
- **Code Generation** - Write and execute code
- **Multi-step Workflows** - Complex tasks requiring multiple tools

### Limitations:

- Experimental and can fail
- Can be slow (multiple LLM calls)
- May choose wrong tools
- Requires good tool descriptions
